In [1]:
%pip install -q langchain-openai langchain-core

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os, json, copy
from typing import Any
from pathlib import Path
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool




MODEL_NAME = "gpt-oss-20b"
YANDEX_CLOUD_FOLDER = "XXX"
API_KEY = "XXX"
os.environ["OPENAI_API_KEY"] = API_KEY

llm = ChatOpenAI(model=f"gpt://{YANDEX_CLOUD_FOLDER}/{MODEL_NAME}",
                temperature=0,
                base_url="https://ai.api.cloud.yandex.net/v1")

def llm_chat(messages: list, tools: list | None = None) -> AIMessage:
    """
    Sends the message history to the LLM and returns the model response.

    Parameters:
      messages — list of dialog messages. Each message is a LangChain object:
                   SystemMessage(content="...")   — instruction for the model (agent role)
                   HumanMessage(content="...")    — message from the user
                   AIMessage(...)                 — previous model response
                   ToolMessage(content="...", tool_call_id="...") — tool result

      tools   — list of tool descriptions (OpenAI function calling schema or LangChain tools).

    Returns AIMessage:
      msg.content    — text response (str)
      msg.tool_calls — list of tool calls:
                         "name" — tool name
                         "args" — arguments (already parsed dict)
                         "id"   — unique call identifier
    """
    if tools:
        return llm.bind_tools(tools).invoke(messages)
    return llm.invoke(messages)


# Product catalog
CATALOG = [
    {"id": "p1",  "name": "Sony WH-1000XM5",            "category": "headphones", "brand": "Sony",     "price": 349, "color": "black",    "rating": 4.8, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p2",  "name": "Sony WH-CH720N",              "category": "headphones", "brand": "Sony",     "price": 129, "color": "blue",     "rating": 4.4, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p3",  "name": "Bose QuietComfort Ultra",     "category": "headphones", "brand": "Bose",     "price": 379, "color": "white",    "rating": 4.7, "tags": ["wireless", "noise-cancelling", "premium"]},
    {"id": "p4",  "name": "Apple AirPods Pro 2",         "category": "earbuds",    "brand": "Apple",    "price": 249, "color": "white",    "rating": 4.6, "tags": ["wireless", "noise-cancelling", "ios"]},
    {"id": "p5",  "name": "Anker Soundcore Liberty 4 NC","category": "earbuds",    "brand": "Anker",    "price": 99,  "color": "black",    "rating": 4.3, "tags": ["wireless", "budget", "noise-cancelling"]},
    {"id": "p6",  "name": "Logitech MX Master 3S",       "category": "mouse",      "brand": "Logitech", "price": 109, "color": "graphite", "rating": 4.8, "tags": ["wireless", "productivity", "premium"]},
    {"id": "p7",  "name": "Logitech Pebble 2",           "category": "mouse",      "brand": "Logitech", "price": 34,  "color": "white",    "rating": 4.2, "tags": ["wireless", "budget", "portable"]},
    {"id": "p8",  "name": "Keychron K2",                 "category": "keyboard",   "brand": "Keychron", "price": 89,  "color": "black",    "rating": 4.5, "tags": ["wireless", "mechanical", "compact"]},
    {"id": "p9",  "name": "NuPhy Air75",                 "category": "keyboard",   "brand": "NuPhy",    "price": 139, "color": "gray",     "rating": 4.6, "tags": ["wireless", "mechanical", "low-profile"]},
    {"id": "p10", "name": "Amazon Kindle Paperwhite",    "category": "ereader",    "brand": "Amazon",   "price": 149, "color": "black",    "rating": 4.7, "tags": ["reading", "portable", "gift"]},
]


@dataclass
class ShopState:
    """Session state: cart and last search results."""
    cart: list = field(default_factory=list)
    last_results: list = field(default_factory=list)


@dataclass
class ToolCallRecord:
    name: str
    args: dict
    result: Any = None


class ToolTracer:
    """Collects all tool calls."""
    def __init__(self):
        self.calls: list[ToolCallRecord] = []

    def record(self, name: str, args: dict, result: Any = None) -> None:
        self.calls.append(ToolCallRecord(name=name, args=args, result=result))

    def called(self, name: str) -> bool:
        return any(c.name == name for c in self.calls)

    def get_calls(self, name: str) -> list:
        return [c for c in self.calls if c.name == name]

    def print_trace(self) -> None:
        print("=== Tool Call Trace ===")
        for i, c in enumerate(self.calls, 1):
            print(f"  {i}. {c.name}({json.dumps(c.args, ensure_ascii=False)[:80]})")
            if c.result is not None:
                print(f"     -> {json.dumps(c.result, ensure_ascii=False)[:100]}")
        print("=====================")


class ShopTools:
    """Shop logic — search and add to cart."""
    def __init__(self, catalog):
        self.catalog = catalog

    def search_products(self, query: str = "", category: str | None = None,
                        brand: str | None = None, max_price: float | None = None,
                        sort_by: str | None = None) -> list:
        results = []
        q_words = query.lower().split() if query else []
        for item in self.catalog:
            hay = f"{item['name']} {item['category']} {item['brand']} {' '.join(item['tags'])}".lower()
            if q_words and not all(w in hay for w in q_words): continue
            if category and item["category"] != category: continue
            if brand and item["brand"].lower() != brand.lower(): continue
            if max_price is not None and item["price"] > float(max_price): continue
            results.append(copy.deepcopy(item))
        if sort_by == "price_asc": results.sort(key=lambda x: x["price"])
        elif sort_by == "rating_desc": results.sort(key=lambda x: -x["rating"])
        return results

    def add_to_cart(self, state: ShopState, product_id: str, quantity: int = 1) -> dict:
        product = next((p for p in self.catalog if p["id"] == product_id), None)
        if not product:
            return {"ok": False, "error": f"Product {product_id} not found"}
        existing = next((r for r in state.cart if r["product_id"] == product_id), None)
        if existing:
            existing["quantity"] += quantity
        else:
            state.cart.append({"product_id": product_id, "name": product["name"],
                                "price": product["price"], "quantity": quantity})
        return {"ok": True, "cart_size": len(state.cart)}


@dataclass
class AgentContext:
    query: str
    max_price: float | None = None
    candidates: list[dict] = field(default_factory=list)
    pros: dict[str, str] = field(default_factory=dict)   # product_id -> pros description
    cons: dict[str, str] = field(default_factory=dict)   # product_id -> cons description
    best: dict | None = None
    cart_result: dict | None = None


TOOLS = ShopTools(CATALOG)
print("Template loaded.")
print(f"  Model: {MODEL_NAME}")
print(f"  Catalog: {len(CATALOG)} products")
print(f"  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool")
print(f"  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage")


Template loaded.
  Model: gpt-oss-20b
  Catalog: 10 products
  Utilities: AgentContext, ToolTracer, ShopTools, convert_to_openai_tool
  LangChain: HumanMessage, SystemMessage, AIMessage, ToolMessage


In [ ]:
def search_products(
    query: str = "",
    category: str | None = None,
    brand: str | None = None,
    max_price: float | None = None,
    sort_by: str | None = None,
) -> list:
    """
    Search for products available in the catalog by given filters.

    Args:
        query: product name (e.g. "Apple AirPods Pro 2")
        category: product category - one of [headphones, earbuds, mouse, keyboard, ereader]
        brand: product brand (e.g. "Apple", "Sony")
        max_price: Price filter - everything that is more than max_price will be excluded from results
        sort_by:  one of "price_asc" - results will be sorted in ascending order of price,
                         "rating_desc" - results will be sorted in descending order of rating
    
    Returns:
        results: List of found items
    """
    ...

def add_to_cart(product_id: str, quantity: int = 1) -> dict:
    """
    Add the product to the shopping cart by its id.

    Args:
        product_id: id of the product
        quantity: the number of added products
    
    Returns:
        dict with cart size or error if nothing was found
    """
    ...

SHOP_TOOLS_SCHEMA = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
]

def run_shopping_agent(user_message: str, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> str:
    """
    ReAct shop agent. Receives a user message and iteratively:
      1. Calls the LLM with the history and tool schema.
      2. If the LLM returns tool_calls — executes each tool:
           search_products -> saves result to state.last_results, records in tracer
           add_to_cart     -> adds product to state.cart, records in tracer
         Adds a ToolMessage with the result to the history and repeats the loop.
      3. If tool_calls is empty — returns the text response from the LLM.
    """

    system_prompt = SystemMessage(content="You are a helpful shopping assistant."
     "Use the provided tools to search for items and add them to cart."
     "Always use tools when the user asks to find or add products."
    "When searching, use appropriate filters based on user's request.")

    

    messages = [system_prompt, HumanMessage(content=user_message)]
    while True:
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content
        
        for tc in ai_msg.tool_calls:
            name = tc['name']
            args = tc['args']
            call_id = tc['id']

            if name == 'search_products':
                result = tools.search_products(**args)
                state.last_results = result
                tracer.record(name, args, result)
            elif name == 'add_to_cart':
                result = tools.add_to_cart(state, **args)
                tracer.record(name, args, result)
            else:
                result = {'error': f'Unknown tool: {name}'}

            messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id))


In [ ]:
PROFILE_PATH = Path("user_profile.json")


def load_profile(path: Path = PROFILE_PATH) -> dict:
    """Loads profile from JSON. Returns {} if file does not exist."""
    if path.exists():
        try:
            return json.loads(path.read_text())
        except json.JSONDecodeError:
            save_profile({})
            return {}
    return {}

def save_profile(profile: dict, path: Path = PROFILE_PATH) -> None:
    """Saves the profile dict to a file as JSON."""
    path.write_text(json.dumps(profile, indent=2, ensure_ascii=False))


def update_profile(key:str, value:str):
    """
    Update a field in the users's persistent profile.

    Recommended field names: name, brand, max_price, color,
    category.

    Args:
        key: Field name to update (e.g. 'brand_preference')
        value: New value for the field (e.g. 'Apple')

    Returns:
        Confirmation message.
    """
    ...

SHOP_TOOLS_SCHEMA_WITH_MEMORY = [
    convert_to_openai_tool(search_products),
    convert_to_openai_tool(add_to_cart),
    convert_to_openai_tool(update_profile),
]


def run_memory_agent(
    user_message: str,
    state: ShopState,
    tools: ShopTools,
    tracer: ToolTracer,
    history: list,
    profile_path: Path = PROFILE_PATH,
) -> tuple:
    """
    Memory agent. Extends run_shopping_agent with long-term and short-term memory.

    Long-term memory:
      - Loads profile from file (load_profile) on each run
      - Passes profile to agent via SystemMessage
      - update_profile tool updates the profile on disk when preferences are first mentioned

    Short-term memory:
      - history contains the full message history from previous turns (including ToolMessages)
      - This allows the agent to "see" the results of past searches in the next turn
      - Added to the query before calling the LLM

    Returns (response: str, updated_history: list).
    Hint: save ALL messages to history (HumanMessage, AIMessage, ToolMessage),
    so the agent knows what was found in the next turn.
    """
    profile = load_profile(profile_path)
    profile_str = json.dumps(profile) if profile else "empty"

    system_prompt = SystemMessage(content="You are a helpful shopping assistant."
     "At the start of each conversation, you will be given the passenger's current profile."
     "Use this information to personalize your responses."

     "Use the provided tools to search for items and add them to cart."
     "Always use tools when the user asks to find or add products."
     "When searching, use appropriate filters based on user's request."

     "RULE: Whenever the passenger tells you their name, prefered brand, maximum price, favourite color, product category,"
     "or any personal detail, you MUST immediately call update_profile to save it."
     "Call it once per field. Do not ask for confirmation — just save it."
     "Recommended profile fields: name, brand, max_price, color, category."
     f"Current user's profile: {profile_str}")

    history.append(HumanMessage(content=user_message))

    messages = [system_prompt] + history

    while True:
        ai_msg = llm_chat(messages, tools=SHOP_TOOLS_SCHEMA_WITH_MEMORY)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg.content, history
        
        for tc in ai_msg.tool_calls:
            name = tc['name']
            args = tc['args']
            call_id = tc['id']

            if name == 'search_products':
                result = tools.search_products(**args)
                state.last_results = result
                tracer.record(name, args, result)
            elif name == 'add_to_cart':
                result = tools.add_to_cart(state, **args)
                tracer.record(name, args, result)
            elif name == 'update_profile':
                profile = load_profile(profile_path)
                key = args['key']
                value = args['value']
                profile[key] = value
                save_profile(profile, profile_path)
                result = {'ok': True, 'key': key, 'value': value}
                tracer.record(name, args, result)
            else:
                result = {'error': f'Unknown tool: {name}'}
            
            tool_msg = ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id)

            messages.append(tool_msg)
            history.append(tool_msg)




In [ ]:
@dataclass
class AgentResult:
    response: str
    trace: list
    context: AgentContext


class RetrieverAgent:
    def run(self, ctx: AgentContext, state: ShopState, tools: ShopTools, tracer: ToolTracer) -> AgentContext:
        """Searches for products via LLM+tools. Fills ctx.candidates and ctx.max_price."""
        system_prompt = SystemMessage(content="You are a electronic store expert. Your task us to find products matching user's criterea and fill max_price"
                                              "Search for up to 5 relevant products via search_products tool"
                                              "Do not fabricate data. Always use search_products")

        messages = [system_prompt] + [HumanMessage(content=ctx.query)]
        while True:
            ai_msg = llm_chat(messages, tools=[convert_to_openai_tool(search_products)])
            if not ai_msg.tool_calls:
                break
            
            for tc in ai_msg.tool_calls:
                name = tc['name']
                args = tc['args']
                call_id = tc['id']

                if name == 'search_products':
                    result = tools.search_products(**args)
                    state.last_results = result
                    ctx.candidates = result[:5]
                    if 'max_price' in args and args['max_price'] is not None:
                        ctx.max_price = float(args['max_price'])
                    tracer.record(name, args, result)
                else:
                    result = {'error': f'Unknown tool: {name}'}
            messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call_id))
        return ctx




class ProsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds pros for each product via LLM. Fills ctx.pros."""
        if not ctx.candidates:
            return ctx
        
        candidates_desc = json.dumps(ctx.candidates, ensure_ascii=False)
        system_prompt = SystemMessage(content="You are a electronic store consultant. You are provided with the list of found products matching user's criterea."
                                              "Your job is to write one-two sentences of pros for each product"
                                              "Do not write cons or any other analysis, only pros"
                                              "For each product respond in the format:\n"
                                              "<product_id>: <pros_description>\n"
                                              "One product per line.\n\n"
                                              f"Products: {candidates_desc}")
        ai_msg = llm_chat([system_prompt])

        for candidate in ctx.candidates:
            pid = candidate['id']
            for line in ai_msg.content.split('\n'):
                if pid in line:
                    parts = line.split(':', 1)
                    if len(parts) > 1:
                        ctx.pros[pid] = parts[1].strip()
                    break
            if pid not in ctx.pros:
                ctx.pros[pid] = 'Pros are unknown'
        tracer.record('analyze_pros', {'candidates': ctx.candidates}, ctx.pros)
        return ctx



class ConsAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Finds cons for each product via LLM. Fills ctx.cons."""
        if not ctx.candidates:
            return ctx
        
        candidates_desc = json.dumps(ctx.candidates, ensure_ascii=False)
        system_prompt = SystemMessage(content="You are a electronic store consultant. You are provided with the list of found products matching user's criterea."
                                              "Your job is to write one-two sentences of cons for each product"
                                              "Do not write cons or any other analysis, only cons"
                                              "For each product respond in the format:\n"
                                              "<product_id>: <pros_description>\n"
                                              "One product per line.\n\n"
                                              f"Products: {candidates_desc}")
        ai_msg = llm_chat([system_prompt])

        for candidate in ctx.candidates:
            pid = candidate['id']
            for line in ai_msg.content.split('\n'):
                if pid in line:
                    parts = line.split(':', 1)
                    if len(parts) > 1:
                        ctx.cons[pid] = parts[1].strip()
                    break
            if pid not in ctx.pros:
                ctx.cons[pid] = 'Cons are unknown'
        tracer.record('analyze_cons', {'candidates': ctx.candidates}, ctx.cons)
        return ctx

class RankerAgent:
    def run(self, ctx: AgentContext, tracer: ToolTracer) -> AgentContext:
        """Picks the best product from ctx.candidates considering ctx.max_price. Fills ctx.best."""
        candidates = ctx.candidates

        if not candidates:
            tracer.record('rank_candidates', {'max_price': ctx.max_price}, None)

        if ctx.max_price:
            candidates = [i for i in candidates if i['price'] <= ctx.max_price]
        candidates = sorted(candidates, key=lambda x: (-x['rating'], x['price']))
        ctx.best = candidates[0]
        tracer.record('rank_candidates', {'max_price': ctx.max_price, 'candidates': ctx.candidates}, ctx.best)
        return ctx        


class CoordinatorAgent:
    def __init__(self):
        self.retriever = RetrieverAgent()
        self.pros_agent = ProsAgent()
        self.cons_agent = ConsAgent()
        self.ranker = RankerAgent()

    def run(self, user_message: str, state: ShopState, tools: ShopTools) -> AgentResult:
        """Orchestrates agents. Returns AgentResult with response, trace, and context."""
        tracer = ToolTracer()
        trace = []

        ctx = AgentContext(query=user_message)


        self.retriever.run(ctx, state, tools, tracer)
        trace.append('delegate_retriever')

        self.pros_agent.run(ctx, tracer)
        trace.append('delegate_pros')

        self.cons_agent.run(ctx, tracer)
        trace.append('delegate_cons')

        self.ranker.run(ctx, tracer)
        trace.append('delegate_ranker')

        add_to_cart_keywords = ['buy', 'purchase', 'add', 'order']
        should_add = any(kw in user_message.lower() for kw in add_to_cart_keywords)

        if should_add and ctx.best:
            cart_res = tools.add_to_cart(state, ctx.best['id'], 1)
            ctx.cart_result = cart_res
            trace.append('delegate_cart')

        response = ''
        if ctx.best:
            best = ctx.best
            response += f"Best product: {best['name']}\n"
            response += f"Price: {best['price']}\n"
            response += f"Rating: {best['rating']}\n"
            response += f"Pros: {ctx.pros[best['id']]}\n"
            response += f"Cons: {ctx.cons[best['id']]}\n"
        else:
            response += 'Nothing was found'


        result = AgentResult(response=response, trace=trace, context=ctx)
        return result

In [28]:
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
print(_res3a.response)

Best product: Logitech MX Master 3S
Price: 109
Rating: 4.8
Pros: The Logitech MX Master 3S offers exceptional precision with its high‑resolution sensor, ergonomic design for extended comfort, customizable buttons for productivity, and a long‑lasting battery that supports quick charging.
Cons: The high price may not justify the incremental features for users who already own a similar mouse. The large size can be uncomfortable for users with smaller hands or those who prefer a more compact design.



In [ ]:
# [1.A] Search with price filter
_s1a = ShopState(); _t1a = ToolTracer()
_r1a = run_shopping_agent("Find wireless headphones under 150 dollars", _s1a, TOOLS, _t1a)
_t1a.print_trace()
assert _t1a.called("search_products"), "FAIL: search_products was not called"
assert all(p["price"] <= 150 for p in _s1a.last_results)
print("OK 1.A")

# [1.B] Search + add the cheapest
_s1b = ShopState(); _t1b = ToolTracer()
_r1b = run_shopping_agent(
    "Find a wireless mouse under 120 dollars and add the cheapest one to cart",
    _s1b, TOOLS, _t1b
)
_t1b.print_trace()
assert _t1b.called("search_products") and _t1b.called("add_to_cart")
assert len(_s1b.cart) == 1 and _s1b.cart[0]["product_id"] == "p7"
print("OK 1.B")

# [1.C] Best keyboard
_s1c = ShopState(); _t1c = ToolTracer()
_r1c = run_shopping_agent(
    "Find a wireless keyboard with the best rating and add it to cart",
    _s1c, TOOLS, _t1c
)
_t1c.print_trace()
assert _t1c.called("search_products") and _t1c.called("add_to_cart")
added = next(p for p in CATALOG if p["id"] == _s1c.cart[0]["product_id"])
assert added["category"] == "keyboard"
print(f"OK 1.C: '{added['name']}' (rating {added['rating']})")


=== Tool Call Trace ===
  1. search_products({"category": "headphones", "max_price": 150, "sort_by": "price_asc"})
     -> [{"id": "p2", "name": "Sony WH-CH720N", "category": "headphones", "brand": "Sony", "price": 129, "co
OK 1.A
=== Tool Call Trace ===
  1. search_products({"category": "mouse", "max_price": 120, "sort_by": "price_asc"})
     -> [{"id": "p7", "name": "Logitech Pebble 2", "category": "mouse", "brand": "Logitech", "price": 34, "c
  2. add_to_cart({"product_id": "p7", "quantity": 1})
     -> {"ok": true, "cart_size": 1}
OK 1.B
=== Tool Call Trace ===
  1. search_products({"category": "keyboard", "sort_by": "rating_desc"})
     -> [{"id": "p9", "name": "NuPhy Air75", "category": "keyboard", "brand": "NuPhy", "price": 139, "color"
  2. add_to_cart({"product_id": "p9", "quantity": 1})
     -> {"ok": true, "cart_size": 1}
OK 1.C: 'NuPhy Air75' (rating 4.6)


In [ ]:
# [2.A] Saving preferences
_p2a = Path("_demo_profile_2a.json")
if _p2a.exists(): _p2a.unlink()
_s2a = ShopState(); _t2a = ToolTracer(); _h2a = []
_r2a, _h2a = run_memory_agent(
    "My name is Anna, I prefer Sony and my budget is 200 dollars",
    _s2a, TOOLS, _t2a, _h2a, _p2a
)
_prof2a = load_profile(_p2a)
print(_prof2a)
assert _t2a.called("update_profile"), "FAIL: update_profile was not called"
assert _t2a.called("update_profile") and _prof2a.get("brand") == "Sony"
print("OK 2.A")

# [2.B] New session uses profile (history=[])
_p2b = Path("_demo_profile_2b.json")
save_profile({"name": "Boris", "brand": "Logitech", "max_price": "150"}, _p2b)
_s2b = ShopState(); _t2b = ToolTracer(); _h2b = []
_r2b, _ = run_memory_agent("What is my name and what is my budget?", _s2b, TOOLS, _t2b, _h2b, _p2b)
print(_r2b)
assert "Boris" in _r2b
print("OK 2.B")

# [2.C] Short-term memory — turn 2 remembers turn 1
_p2c = Path("_demo_profile_2c.json")
if _p2c.exists(): _p2c.unlink()
_s2c = ShopState(); _h2c = []
_, _h2c = run_memory_agent(
    "Find wireless headphones under 150 dollars", _s2c, TOOLS, ToolTracer(), _h2c, _p2c
)
assert len(_h2c) >= 2
_t2c2 = ToolTracer()
_, _h2c = run_memory_agent(
    "Add the first one found to cart", _s2c, TOOLS, _t2c2, _h2c, _p2c
)
assert _t2c2.called("add_to_cart") and len(_s2c.cart) == 1
print(f"OK 2.C: added '{_s2c.cart[0]['name']}'")


{'name': 'Anna', 'brand': 'Sony', 'max_price': '200'}
OK 2.A
Your name is **Boris** and your budget is **$150**.
OK 2.B
OK 2.C: added 'Sony WH-CH720N'


In [ ]:
# [3.A] Full cycle: search -> pros -> cons -> ranking -> cart
_s3a = ShopState()
_res3a = CoordinatorAgent().run(
    "Find the best wireless mouse under 120 dollars and add it to cart", _s3a, TOOLS
)
assert "delegate_retriever" in _res3a.trace
assert "delegate_pros" in _res3a.trace and "delegate_cons" in _res3a.trace
assert "delegate_ranker" in _res3a.trace and "delegate_cart" in _res3a.trace
assert len(_s3a.cart) == 1 and _s3a.cart[0]["product_id"] == "p6"
assert _res3a.context.best is not None and _res3a.context.best["id"] == "p6"
assert len(_res3a.context.pros) > 0 and len(_res3a.context.cons) > 0
print("OK 3.A")

# [3.B] Search only, no add to cart
_s3b = ShopState()
_res3b = CoordinatorAgent().run("Find a wireless keyboard", _s3b, TOOLS)
assert "delegate_retriever" in _res3b.trace
assert "delegate_pros" in _res3b.trace and "delegate_cons" in _res3b.trace
assert "delegate_ranker" in _res3b.trace
assert "delegate_cart" not in _res3b.trace and len(_s3b.cart) == 0
assert _res3b.context.best is not None
print("OK 3.B")

# [3.C] RankerAgent — price tie-break with equal rating
_ctx3c = AgentContext(query="test", candidates=[
    {"id": "x1", "name": "A", "price": 200, "rating": 4.8},
    {"id": "x2", "name": "B", "price": 150, "rating": 4.8},
    {"id": "x3", "name": "C", "price": 100, "rating": 4.5},
])
_tr3c = ToolTracer()
_ctx3c = RankerAgent().run(_ctx3c, _tr3c)
assert _ctx3c.best["id"] == "x2" and _tr3c.called("rank_candidates")
print("OK 3.C")

# [3.D] RankerAgent respects ctx.max_price
_ctx3d = AgentContext(
    query="mouse under 120 dollars",
    max_price=120.0,
    candidates=[
        {"id": "expensive", "name": "Super Mouse",  "price": 200, "rating": 4.9},
        {"id": "p6",        "name": "MX Master 3S", "price": 109, "rating": 4.8},
        {"id": "p7",        "name": "Pebble 2",      "price": 34,  "rating": 4.2},
    ],
)
_tr3d = ToolTracer()
_ctx3d = RankerAgent().run(_ctx3d, _tr3d)
assert _ctx3d.best is not None and _ctx3d.best["id"] == "p6"
print("OK 3.D: context passed correctly, max_price is respected")


OK 3.A
OK 3.B
OK 3.C
OK 3.D: context passed correctly, max_price is respected
